# Laboratorio 4: Análisis de Datos GeoEspaciales

CC3084 – Data Science

Iris Ayala - Anggie Quezada - Jonathan Diaz

Análisis de la proliferación de cianobacteria en los lagos de Atitlán y Amatitlán usando imágenes Sentinel-2.

## Librerías

In [1]:
import os
import openeo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from datetime import date, timedelta
from sentinelhub import SHConfig, SentinelHubRequest, DataCollection, MimeType, CRS, BBox, bbox_to_dimensions

## 1. Conexión con la API de Sentinel-2

In [4]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=BKWR-BPKY 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


## 2. Obtención de los datos raster

### Coordenadas de los lagos

In [5]:
lago_atitlan = {
    "west": -91.326256,
    "east": -91.07151,
    "south": 14.5948,
    "north": 14.750979
}

lago_amatitlan = {
    "west": -90.638065,
    "east": -90.512924,
    "south": 14.412347,
    "north": 14.493799
}

### Fechas oficiales por lago

In [6]:
fechas_atitlan = [
    "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17", "2025-11-21",
    "2025-12-29", "2026-02-12", "2026-03-24", "2026-04-13", "2026-04-28", "2026-07-22"
]

fechas_amatitlan = [
    "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24", "2026-01-08",
    "2026-02-02", "2026-02-07", "2026-03-29", "2026-04-13", "2026-04-28", "2026-06-19"
]

### Descarga de bandas

Se descargan únicamente las bandas B03, B04 y B08, necesarias para calcular NDVI y NDWI.

In [9]:
def descargar_bandas(bbox, fecha, ruta_salida):
    fecha_fin = (date.fromisoformat(fecha) + timedelta(days=1)).isoformat()
    cubo = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[fecha, fecha_fin],
        bands=["B03", "B04", "B08"]
    )
    cubo.download(ruta_salida, format="GTIFF")

In [10]:
os.makedirs("../data/GIS/atitlan", exist_ok=True)
os.makedirs("../data/GIS/amatitlan", exist_ok=True)

for fecha in fechas_atitlan:
    ruta = f"../data/GIS/atitlan/{fecha}.tif"
    descargar_bandas(lago_atitlan, fecha, ruta)

for fecha in fechas_amatitlan:
    ruta = f"../data/GIS/amatitlan/{fecha}.tif"
    descargar_bandas(lago_amatitlan, fecha, ruta)

KeyboardInterrupt: 

### Verificación de una imagen descargada

In [ ]:
with rasterio.open(f"../data/GIS/atitlan/{fechas_atitlan[0]}.tif") as src:
    bandas = src.read()
    print("Número de bandas:", src.count)
    print("Dimensiones:", src.width, src.height)

## 3. Índice de cianobacteria, NDVI y NDWI

### 3.1 Configuración de Sentinel Hub

In [ ]:
import os
from dotenv import load_dotenv
from sentinelhub import SHConfig

load_dotenv()

config = SHConfig()
config.sh_client_id = os.environ["SH_CLIENT_ID"]
config.sh_client_secret = os.environ["SH_CLIENT_SECRET"]

### 3.2 Script de detección de cianobacteria

Se usa el script "Cyanobacteria Chlorophyll-a NDCI L1C" de https://custom-scripts.sentinel-hub.com. El script original devuelve un color RGB para visualizar la floración. Se adapta la salida para que devuelva el valor numérico de clorofila-a (`chl`) en lugar del color, manteniendo la misma lógica de detección de agua y el mismo modelo NDCI, ya que el análisis temporal y de correlación (Ejercicio 4) necesita valores numéricos.

Valores de salida:
- `0`: pixel que no es agua.
- `-1`: agua con floración superficial detectada (índice de vegetación flotante alto).
- cualquier otro valor: estimado de clorofila-a (proxy de cianobacteria).

In [ ]:
evalscript_cianobacteria = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04", "B05", "B07", "B08", "B8A", "B11", "B12"],
    output: { bands: 1, sampleType: "FLOAT32" }
  };
}

var MNDWI_threshold = 0.42;
var NDWI_threshold = 0.4;
var filter_UABS = true;

function wbi(r, g, b, nir, swir1, swir2) {
  let ws = 0;
  try {
    var ndvi = (nir - r) / (nir + r);
    var mndwi = (g - swir1) / (g + swir1);
    var ndwi = (g - nir) / (g + nir);
    var ndwi_leaves = (nir - swir1) / (nir + swir1);
    var aweish = b + 2.5 * g - 1.5 * (nir + swir1) - 0.25 * swir2;
    var aweinsh = 4 * (g - swir1) - (0.25 * nir + 2.75 * swir1);
    var dbsi = ((swir1 - g) / (swir1 + g)) - ndvi;
    if (mndwi > MNDWI_threshold || ndwi > NDWI_threshold || aweinsh > 0.1879 || aweish > 0.1112 || ndvi < -0.2 || ndwi_leaves > 1) {
      ws = 1;
    }
    if (filter_UABS && ws == 1) {
      if (aweinsh <= -0.03 || dbsi > 0) { ws = 0; }
    }
  } catch (err) { ws = 0; }
  return ws;
}

function FAI(a, b, c) {
  return (b - a - (c - a) * (783 - 665) / (865 - 665));
}

function NDCI(a, b) {
  return (b - a) / (b + a);
}

function evaluatePixel(s) {
  let water = wbi(s.B04, s.B03, s.B02, s.B08, s.B11, s.B12);
  if (water == 0) { return [0]; }

  let FAIv = FAI(s.B04, s.B07, s.B8A);
  if (FAIv > 0.08) { return [-1]; }

  let NDCIv = NDCI(s.B04, s.B05);
  let chl = 826.57 * Math.pow(NDCIv, 3) - 176.43 * Math.pow(NDCIv, 2) + 19 * NDCIv + 4.071;
  return [chl];
}
"""

### Función para obtener el índice de cianobacteria

In [ ]:
def obtener_indice_cianobacteria(bbox_dict, fecha):
    bbox = BBox(bbox=[bbox_dict["west"], bbox_dict["south"], bbox_dict["east"], bbox_dict["north"]], crs=CRS.WGS84)
    tamano = bbox_to_dimensions(bbox, resolution=10)
    solicitud = SentinelHubRequest(
        evalscript=evalscript_cianobacteria,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(fecha, fecha)
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox,
        size=tamano,
        config=config
    )
    return solicitud.get_data()[0]

### Cálculo de NDVI y NDWI a partir de las bandas descargadas

In [ ]:
def calcular_ndvi_ndwi(ruta_tif):
    with rasterio.open(ruta_tif) as src:
        bandas = src.read()
    green = bandas[0].astype(np.float32) / 10000
    red = bandas[1].astype(np.float32) / 10000
    nir = bandas[2].astype(np.float32) / 10000
    ndvi = np.where((nir + red) == 0, 0, (nir - red) / (nir + red))
    ndwi = np.where((green + nir) == 0, 0, (green - nir) / (green + nir))
    return ndvi, ndwi

### Cálculo para todas las fechas de cada lago

In [ ]:
resultados_atitlan = {}
for fecha in fechas_atitlan:
    ruta = f"../data/GIS/atitlan/{fecha}.tif"
    ndvi, ndwi = calcular_ndvi_ndwi(ruta)
    cianobacteria = obtener_indice_cianobacteria(lago_atitlan, fecha)
    resultados_atitlan[fecha] = {"ndvi": ndvi, "ndwi": ndwi, "cianobacteria": cianobacteria}

In [ ]:
resultados_amatitlan = {}
for fecha in fechas_amatitlan:
    ruta = f"../data/GIS/amatitlan/{fecha}.tif"
    ndvi, ndwi = calcular_ndvi_ndwi(ruta)
    cianobacteria = obtener_indice_cianobacteria(lago_amatitlan, fecha)
    resultados_amatitlan[fecha] = {"ndvi": ndvi, "ndwi": ndwi, "cianobacteria": cianobacteria}

### Visualización del índice de cianobacteria

In [ ]:
fecha_ejemplo_atitlan = fechas_atitlan[0]
plt.figure(figsize=(7, 6))
plt.imshow(resultados_atitlan[fecha_ejemplo_atitlan]["cianobacteria"], cmap="YlOrRd")
plt.title(f"Índice de cianobacteria - Atitlán - {fecha_ejemplo_atitlan}")
plt.colorbar(label="Clorofila-a estimada")
plt.axis("off")
plt.show()

In [ ]:
fecha_ejemplo_amatitlan = fechas_amatitlan[0]
plt.figure(figsize=(7, 6))
plt.imshow(resultados_amatitlan[fecha_ejemplo_amatitlan]["cianobacteria"], cmap="YlOrRd")
plt.title(f"Índice de cianobacteria - Amatitlán - {fecha_ejemplo_amatitlan}")
plt.colorbar(label="Clorofila-a estimada")
plt.axis("off")
plt.show()